# MiSTR for flood tracking on GEOID-Flood

A sequence of 32 Sentinel-1 passes (same orbit, 512 x 512 px at 10 m, VV/VH) around a mapped flood;
the model labels every pass as dry / permanent water / flood (causal: a pass only sees earlier
ones). Fixed resolution encodes every pass at one patch size; MiSTR encodes all passes at 128 px
patches, then re-encodes the 32 px windows its selector ranks highest (budget q: a share of all
windows of the series) at 32 px.

Needs: `checkpoints/geoidchange/{trkO_128c32_qr_s0,trkO_fix128_cont_s0,trkO_fix32_cont_s0}.pt` and
`demo/geoid_test_series.npz` (two test series, the event at pass 16; see the README, *Demo*). A GPU.

In [ ]:
import os, sys
# run from the repo root (the notebooks import the repo's packages)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
assert torch.cuda.is_available(), "the models run on a GPU (OlmoEarth's FlashAttention path)"

In [ ]:
from exp.demo import load_geoid, load_geoid_samples, predict_geoid
from exp.viz.geoid_common import miou, CMAP

names, items = load_geoid_samples("demo/geoid_test_series.npz")
C = "checkpoints/geoidchange"
models = {"fixed S128": (load_geoid(f"{C}/trkO_fix128_cont_s0.pt", 128), 0.0),
          "MiSTR S128 & S32, q 8 %": (load_geoid(f"{C}/trkO_128c32_qr_s0.pt", 128, 32), 0.08),
          "fixed S32": (load_geoid(f"{C}/trkO_fix32_cont_s0.pt", 32), 0.0)}
res = {(n, m): predict_geoid(net, it, q) for n, it in zip(names, items) for m, (net, q) in models.items()}
for (n, m), o in res.items():
    k = o["event"]
    print(f"{n:16} {m:24} 3-class mIoU on the labelled pass {100 * miou(o['pred'][k - 1], o['label']):5.1f}"
          f"   tokens {o['tokens']:5d}")

The paper scores 1,000+ test series with the crop budget shared across a batch of 8 series
(so flood-free series spend little); here each series has its own budget, so MiSTR's numbers
differ slightly from the paper's per-series values.

Passes around the labelled date (VV backscatter; label outlines on its date; predictions per model):

In [ ]:
show = range(-3, 4)            # passes around the labelled one (pass k; prediction for pass t = pred[t - 1])
for n, it in zip(names, items):
    o0 = res[(n, "fixed S128")]
    k, days = o0["event"], o0["days"]
    fig, axes = plt.subplots(1 + len(models), len(show), figsize=(1.6 * len(show), 1.7 * (1 + len(models))))
    for j, dk in enumerate(show):
        t = k + dk
        axes[0, j].imshow(it[0][t, 0], cmap="gray", vmin=-25, vmax=0)      # VV, dB
        axes[0, j].set_title("labelled pass" if dk == 0 else f"{days[t]:+.0f} d", fontsize=8)
        if dk == 0:
            axes[0, j].contour(o0["label"] == 2, levels=[0.5], colors="cyan", linewidths=0.5)
        for r, m in enumerate(models, 1):
            o = res[(n, m)]
            axes[r, j].imshow(o["pred"][t - 1] + 1, cmap=CMAP, vmin=0, vmax=3, interpolation="nearest")
            if o["windows"] is not None:
                sel = o["windows"]["sel_t"] == t
                for y, x in zip(o["windows"]["win_y"][sel], o["windows"]["win_x"][sel]):
                    axes[r, j].add_patch(Rectangle((x - 0.5, y - 0.5), 32, 32, fill=False, lw=0.4, ec="w"))
            if j == 0:
                axes[r, j].set_ylabel(m, fontsize=7)
    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(n, fontsize=9); plt.tight_layout()